# AstroCLIMB: memory-safe Kaggle starter

This notebook never loads either 10 GB CSV in full. It streams a few rows at a time, turns captions and images into CLIP embeddings, saves compact feature shards, trains a small classifier, and creates `submission.csv`.

Recommended Kaggle settings: **GPU accelerator**, persistence enabled, and Internet enabled for the first run if the CLIP model is not attached as a Kaggle Model/Dataset. A complete feature-extraction run can be saved as a Kaggle Dataset so later experiments do not need to parse the CSVs again.

In [ ]:
# Usually already installed on Kaggle. Uncomment only if the import below fails.
# !pip install -q transformers sentencepiece

In [ ]:
import base64, gc, hashlib, io, json, os, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageFile
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from transformers import CLIPModel, CLIPProcessor

ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings('ignore', message='.*DecompressionBomb.*')

LABELS = ['same_figure', 'same_paper', 'related_papers', 'unrelated_papers']
WORK = Path('/kaggle/working/astroclimb_features')
WORK.mkdir(parents=True, exist_ok=True)
CSV_CHUNK_SIZE = 8       # raw Base64 rows held at once; lower to 2-4 if RAM spikes
MODEL_BATCH_SIZE = 8     # lower if GPU runs out of memory
MAX_TEXT_LENGTH = 77     # native CLIP context length
SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

## Locate the competition files

The next cell searches all attached Kaggle inputs. If several datasets contain files with the same name, replace the selected paths manually.

In [ ]:
def find_file(filename):
    matches = sorted(Path('/kaggle/input').glob(f'**/{filename}'))
    if not matches:
        raise FileNotFoundError(f'Could not find {filename} under /kaggle/input')
    print(filename, 'matches:', [str(p) for p in matches])
    return matches[0]

TRAIN_CSV = find_file('train.csv')
TEST_CSV = find_file('test.csv')
SAMPLE_CSV = find_file('sample_submission.csv')
print('Using:', TRAIN_CSV, TEST_CSV, SAMPLE_CSV, sep='\n')

## Load CLIP

CLIP places images and captions in one embedding space, which is useful for all three pair modalities. With Internet disabled, attach a downloaded Hugging Face model as a Kaggle Dataset and set `MODEL_NAME` to its local directory.

In [ ]:
MODEL_NAME = 'openai/clip-vit-base-patch32'  # or '/kaggle/input/YOUR-CLIP-MODEL-FOLDER'
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
model = CLIPModel.from_pretrained(MODEL_NAME).eval().to(DEVICE)
if DEVICE == 'cuda':
    model = model.half()
EMBED_DIM = int(model.config.projection_dim)
print('Embedding dimension:', EMBED_DIM)

## Streaming feature extraction

Each object is decoded only inside its small chunk. Pair features are symmetric: absolute embedding difference, elementwise product, cosine similarity, modality indicators, text-length statistics, and exact-text equality. Saved arrays use `float16`; fitting later converts them to `float32`.

In [ ]:
def is_png_b64(value):
    return isinstance(value, str) and value.lstrip().startswith('iVBOR')

def decode_png(value):
    raw = base64.b64decode(value.strip(), validate=False)
    with Image.open(io.BytesIO(raw)) as im:
        im.thumbnail((2048, 2048))
        return im.convert('RGB').copy()

def l2_normalize(x):
    return x / np.clip(np.linalg.norm(x, axis=1, keepdims=True), 1e-8, None)

@torch.inference_mode()
def encode_values(values):
    values = [v if isinstance(v, str) else '' for v in values]
    image_mask = np.array([is_png_b64(v) for v in values], dtype=bool)
    embeddings = np.zeros((len(values), EMBED_DIM), dtype=np.float32)

    text_idx = np.flatnonzero(~image_mask)
    for start in range(0, len(text_idx), MODEL_BATCH_SIZE):
        idx = text_idx[start:start + MODEL_BATCH_SIZE]
        texts = [values[i] for i in idx]
        inputs = processor(text=texts, padding=True, truncation=True,
                           max_length=MAX_TEXT_LENGTH, return_tensors='pt')
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        # Calling the encoder explicitly works across old and new Transformers APIs.
        text_output = model.text_model(**inputs)
        z = model.text_projection(text_output.pooler_output)
        embeddings[idx] = z.float().cpu().numpy()

    image_idx = np.flatnonzero(image_mask)
    for start in range(0, len(image_idx), MODEL_BATCH_SIZE):
        idx = image_idx[start:start + MODEL_BATCH_SIZE]
        images, valid = [], []
        for i in idx:
            try:
                images.append(decode_png(values[i]))
                valid.append(i)
            except Exception as exc:
                print(f'Warning: failed to decode object at local row {i}: {exc}')
        if images:
            inputs = processor(images=images, return_tensors='pt')
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            if DEVICE == 'cuda': inputs['pixel_values'] = inputs['pixel_values'].half()
            vision_output = model.vision_model(pixel_values=inputs['pixel_values'])
            z = model.visual_projection(vision_output.pooler_output)
            embeddings[np.asarray(valid)] = z.float().cpu().numpy()
        del images

    return l2_normalize(embeddings), image_mask

def make_pair_features(chunk):
    a_values = chunk['obj_1'].fillna('').tolist()
    b_values = chunk['obj_2'].fillna('').tolist()
    a, a_img = encode_values(a_values)
    b, b_img = encode_values(b_values)
    cosine = np.sum(a * b, axis=1, keepdims=True)
    modalities = np.column_stack([~a_img & ~b_img, a_img ^ b_img, a_img & b_img]).astype(np.float32)
    len_a = np.array([0 if x else min(len(v), 5000) / 5000 for v, x in zip(a_values, a_img)], dtype=np.float32)[:, None]
    len_b = np.array([0 if x else min(len(v), 5000) / 5000 for v, x in zip(b_values, b_img)], dtype=np.float32)[:, None]
    exact_text = np.array([(not ai and not bi and x.strip() == y.strip())
                           for x, y, ai, bi in zip(a_values, b_values, a_img, b_img)],
                          dtype=np.float32)[:, None]
    extra = np.hstack([cosine, modalities, np.abs(len_a-len_b), np.minimum(len_a, len_b), exact_text])
    return np.hstack([np.abs(a-b), a*b, extra]).astype(np.float16)

def extract_csv(csv_path, split, has_labels):
    out_dir = WORK / split
    out_dir.mkdir(exist_ok=True)
    usecols = ['id', 'obj_1', 'obj_2'] + (LABELS if has_labels else [])
    reader = pd.read_csv(csv_path, usecols=usecols, chunksize=CSV_CHUNK_SIZE)
    started = time.time()
    for part, chunk in enumerate(reader):
        target = out_dir / f'part_{part:05d}.npz'
        if target.exists():
            continue  # resumable after notebook interruption
        X = make_pair_features(chunk)
        arrays = {'ids': chunk['id'].to_numpy(), 'X': X}
        if has_labels:
            arrays['y'] = chunk[LABELS].to_numpy().argmax(axis=1).astype(np.int8)
        np.savez_compressed(target, **arrays)
        if part % 25 == 0:
            print(f'{split}: part {part}, rows through {part*CSV_CHUNK_SIZE + len(chunk)}, elapsed {(time.time()-started)/60:.1f} min')
        del chunk, X, arrays
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# Run train first. If interrupted, rerun; completed shard files are skipped.
extract_csv(TRAIN_CSV, 'train', has_labels=True)

After train extraction, consider using **Save Version** and turning `/kaggle/working/astroclimb_features` into a private Kaggle Dataset. Feature files are far smaller and faster to reuse than the CSV.

In [ ]:
def load_shards(split, with_labels):
    files = sorted((WORK / split).glob('part_*.npz'))
    if not files:
        raise RuntimeError(f'No feature shards found for {split}')
    ids, features, labels = [], [], []
    for path in files:
        with np.load(path) as data:
            ids.append(data['ids'])
            features.append(data['X'])
            if with_labels:
                labels.append(data['y'])
    result = [np.concatenate(ids), np.concatenate(features).astype(np.float32)]
    if with_labels:
        result.append(np.concatenate(labels))
    return tuple(result)

train_ids, X, y = load_shards('train', with_labels=True)
print(X.shape, np.bincount(y, minlength=4))

tr, va = train_test_split(np.arange(len(y)), test_size=0.2, random_state=SEED, stratify=y)
validator = LogisticRegression(C=2.0, class_weight='balanced', max_iter=1500, solver='lbfgs')
validator.fit(X[tr], y[tr])
valid_pred = validator.predict(X[va])
print('Validation macro-F1:', f1_score(y[va], valid_pred, average='macro'))
print(classification_report(y[va], valid_pred, target_names=LABELS, digits=4))

# Random row splitting can be optimistic if objects repeat across pairs. Use this only as a starter check.
classifier = LogisticRegression(C=2.0, class_weight='balanced', max_iter=1500, solver='lbfgs')
classifier.fit(X, y)
del validator, valid_pred
gc.collect()

## Extract test features and write the submission

In [ ]:
# Free training features before parsing the raw test CSV.
del X, y, train_ids
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

extract_csv(TEST_CSV, 'test', has_labels=False)
test_ids, X_test = load_shards('test', with_labels=False)
pred = classifier.predict(X_test)

submission = pd.DataFrame({'id': test_ids})
for class_index, label in enumerate(LABELS):
    submission[label] = (pred == class_index).astype(np.int8)

# Restore sample order and retain only the required prediction columns.
sample = pd.read_csv(SAMPLE_CSV, usecols=lambda c: c == 'id' or c in LABELS)
submission = sample[['id']].merge(submission, on='id', how='left')
assert submission[LABELS].notna().all().all()
assert (submission[LABELS].sum(axis=1) == 1).all()
submission.to_csv('/kaggle/working/submission.csv', index=False)
display(submission.head())
print('Saved /kaggle/working/submission.csv with shape', submission.shape)

## Useful next improvements

- Cache embeddings by a hash of each object when objects recur frequently. Do not use the full Base64 string as a dictionary key.
- Add perceptual hashes for image–image pairs and TF-IDF similarities for caption–caption pairs.
- Group the validation split by object hashes to reduce leakage from repeated objects.
- Tune class-specific decision thresholds using out-of-fold probabilities because the metric is macro-F1.
- Try a nonlinear classifier after the pipeline works; the compact features make experimentation inexpensive.

If memory still rises, reduce `CSV_CHUNK_SIZE` first. The key constraint is the size of raw Base64 strings—not the number of rows.